# LIDC-IDRI CT Data Exploration Pipeline (Google Colab)

Pipeline chuẩn y tế (Ground-Truth-Aware Medical Pipeline) phục vụ khảo sát, tải và tiền xử lý dữ liệu CT từ dataset **LIDC-IDRI** (The Lung Image Database Consortium and Image Database Resource Initiative).

### Định hướng nghiên cứu dài hạn:
`LIDC-IDRI → CT Preprocessing → 2.5D / 3D-aware Representation → MambaVision → Inter-Slice Mamba → Lung Nodule Detection / Segmentation`

### Mục tiêu notebook này:
1. Kết nối Public NBIA API v1 từ The Cancer Imaging Archive (TCIA) không cần token xác thực.
2. Tải duy nhất 1 CT series của bệnh nhân để khảo sát nhanh (tránh tải toàn bộ 133 GB).
3. Đọc DICOM đệ quy, sắp xếp slice theo trục Z (`ImagePositionPatient[2]`), bảo toàn toạ độ `(z, y, x)`.
4. Chuyển đổi sang Hounsfield Units (HU) theo rescale slope/intercept chuẩn của từng slice.
5. Hiển thị Raw HU, Lung Window `[-1000, +400 HU]`, và lưới 12 lát cắt phân bố đều toàn bộ thể tích.
6. Trích xuất annotation nốt phổi (LIDC XML & DICOM SEG/SR), bảo tồn các đặc trưng nốt phổi (subtlety, malignancy, ...) và hiển thị contour / mask ground truth.


---
## 01. Install dependencies
Cài đặt các thư viện cần thiết cho xử lý ảnh y tế và giao tiếp API: `pydicom`, `requests`, `numpy`, `matplotlib`, `tqdm`, `pandas`.


In [ ]:
# 01. Install dependencies
!pip -q install pydicom numpy pandas matplotlib requests tqdm

from pathlib import Path
import json
import zipfile
import re
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import pydicom

print("✅ Thư viện sẵn sàng!")
print("pydicom version:", pydicom.__version__)
print("numpy version  :", np.__version__)


---
## 02. Configuration
Cấu hình các thư mục làm việc trên Colab (`/content/lidc_demo`), các tham số API và các ngưỡng cảnh báo dung lượng.


In [ ]:
# 02. Configuration
COLLECTION = "LIDC-IDRI"
DEFAULT_PATIENT_ID = "LIDC-IDRI-0001"

BASE_DIR = Path("/content/lidc_demo")
RAW_DIR = BASE_DIR / "raw"
EXTRACT_DIR = BASE_DIR / "dicom"
XML_DIR = Path("/content/LIDC_XML")

RAW_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
XML_DIR.mkdir(parents=True, exist_ok=True)

# Windowing presets
LUNG_WINDOW_VMIN = -1000.0
LUNG_WINDOW_VMAX = 400.0

print("Base Dir   :", BASE_DIR)
print("Raw Zip Dir:", RAW_DIR)
print("DICOM Dir  :", EXTRACT_DIR)


---
## 03. TCIA API helper
Hàm `api_get` gọi endpoint của Public NBIA API v1 (`https://services.cancerimagingarchive.net/nbia-api/services/v1`).
- Bắt buộc dùng v1 (không dùng v2 vì v2 yêu cầu bearer token).
- Kiểm tra status code và response text.
- Không gọi `.json()` nếu body rỗng để tránh `JSONDecodeError`.
- In URL, HTTP status, content-type, độ dài response để debug minh bạch.


In [ ]:
# 03. TCIA API helper
BASE_URL = "https://services.cancerimagingarchive.net/nbia-api/services/v1"

def api_get(endpoint: str, params: dict = None, timeout: int = 120, verbose: bool = True):
    url = f"{BASE_URL}/{endpoint}"
    response = requests.get(
        url,
        params=params,
        timeout=timeout,
        headers={"User-Agent": "LIDC-IDRI-Colab-Pipeline/1.0"}
    )

    if verbose:
        print(f"[{endpoint}]")
        print("  URL         :", response.url)
        print("  Status      :", response.status_code)
        print("  Content-Type:", response.headers.get("content-type"))
        print("  Length (B)  :", len(response.text))

    response.raise_for_status()

    if not response.text.strip():
        raise RuntimeError(
            f"API trả HTTP {response.status_code} nhưng response body rỗng cho endpoint '{endpoint}'.\n"
            f"URL thực tế: {response.url}"
        )

    return response

print("✅ api_get helper đã được cấu hình với BASE_URL v1.")


---
## 04. Test API
Kiểm tra kết nối tới TCIA bằng endpoint `getCollectionValues`.


In [ ]:
# 04. Test API
response = api_get("getCollectionValues", timeout=60)
print("\nFirst 300 characters of response:")
print(response.text[:300])
print("\n[PASS] Public API NBIA v1 hoạt động bình thường!")


---
## 05. List collections
Xác minh sự tồn tại của collection `LIDC-IDRI` trong danh sách các dataset y tế công khai của TCIA.


In [ ]:
# 05. List collections
collections_data = response.json()
collections = [item.get("Collection") for item in collections_data if isinstance(item, dict) and "Collection" in item]

print(f"Tổng số public collections trên TCIA: {len(collections)}")
lidc_matches = [c for c in collections if "LIDC" in c.upper()]
print("LIDC collections tìm thấy:", lidc_matches)

assert COLLECTION in collections, f"Không tìm thấy {COLLECTION} trong danh sách collection của TCIA!"
print(f"✅ Xác nhận: '{COLLECTION}' tồn tại trong TCIA.")


---
## 06. Find patients
Truy vấn danh sách bệnh nhân thuộc collection `LIDC-IDRI` bằng endpoint `getPatient`.


In [ ]:
# 06. Find patients
response = api_get("getPatient", params={"Collection": COLLECTION, "format": "json"})
patients = response.json()

print(f"Tổng số bệnh nhân trong {COLLECTION}: {len(patients)}")
print("3 bệnh nhân đầu tiên:")
for p in patients[:3]:
    print(" ", p)

print("\n[PASS] Đã lấy danh sách bệnh nhân thành công.")


---
## 07. Select patient
Chọn bệnh nhân cần khảo sát. Mặc định chọn bệnh nhân đầu tiên: `LIDC-IDRI-0001`.


In [ ]:
# 07. Select patient
PATIENT_INDEX = 0
selected_patient = patients[PATIENT_INDEX]
patient_id = selected_patient["PatientId"]

print(f"Bệnh nhân được chọn [index {PATIENT_INDEX}]: {patient_id}")
print("Patient metadata:")
for k, v in selected_patient.items():
    print(f"  {k:20s}: {v}")


---
## 08. List studies
Truy vấn các ca chụp (Study) của bệnh nhân bằng endpoint `getPatientStudy`.


In [ ]:
# 08. List studies
response = api_get(
    "getPatientStudy",
    params={
        "Collection": COLLECTION,
        "PatientID": patient_id,
        "format": "json"
    }
)
studies = response.json()

print(f"Bệnh nhân {patient_id} có {len(studies)} study:")
for i, st in enumerate(studies):
    print(f"\n[Study {i}]")
    print(f"  StudyInstanceUID: {st.get('StudyInstanceUID')}")
    print(f"  StudyDate       : {st.get('StudyDate')}")
    print(f"  SeriesCount     : {st.get('SeriesCount')}")

print("\n[PASS] Lấy danh sách Study thành công.")


---
## 09. Select study
> ⚠️ **LƯU Ý QUAN TRỌNG VỀ DỮ LIỆU LIDC-IDRI**:
> - `Study 0` chỉ có `SeriesCount = 1`, chứa ảnh X-quang ngực phẳng (**Modality: DX**), KHÔNG PHẢI CT.
> - `Study 1` có `SeriesCount = 9`, chứa ảnh cắt lớp lồng ngực (**Modality: CT** gồm 133 lát cắt) cùng 4 bộ phân vùng nốt phổi DICOM SEG và 4 bản báo cáo bác sĩ DICOM SR.
> - Nếu chọn Study 0 và cố ép tìm CT, API sẽ trả body rỗng. Do đó ta chọn Study 1.


In [ ]:
# 09. Select study
# Chọn Study 1 (Study chứa CT scan và các bộ phân vùng SEG/SR)
SELECTED_STUDY_INDEX = 1

if SELECTED_STUDY_INDEX >= len(studies):
    SELECTED_STUDY_INDEX = 0

selected_study = studies[SELECTED_STUDY_INDEX]
study_uid = selected_study["StudyInstanceUID"]

print(f"Đã chọn Study {SELECTED_STUDY_INDEX}:")
print("  StudyInstanceUID:", study_uid)
print("  SeriesCount     :", selected_study.get("SeriesCount"))


---
## 10. List series
Lấy toàn bộ series trong Study đã chọn bằng endpoint `getSeries`.
**Quy tắc**: Không truyền tham số `Modality=CT` lên API ở bước này, mà lấy toàn bộ series rồi lọc bằng Python để tránh lỗi response rỗng.


In [ ]:
# 10. List series
response = api_get(
    "getSeries",
    params={
        "Collection": COLLECTION,
        "PatientID": patient_id,
        "StudyInstanceUID": study_uid,
        "format": "json"
    }
)

all_series = response.json()
print(f"\nTìm thấy {len(all_series)} series trong Study:")

df_series = []
for i, s in enumerate(all_series):
    size_mb = s.get("FileSize", 0) / (1024**2) if s.get("FileSize") else 0.0
    df_series.append({
        "Index": i,
        "Modality": s.get("Modality"),
        "ImageCount": s.get("ImageCount"),
        "Size_MB": round(size_mb, 2),
        "SeriesDescription": s.get("SeriesDescription"),
        "AnnotationsFlag": s.get("AnnotationsFlag"),
        "SeriesInstanceUID": s.get("SeriesInstanceUID"),
    })

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
print(pd.DataFrame(df_series).to_string(index=False))

print("\n[PASS] Lấy danh sách series thành công.")


---
## 11. Filter CT series
Lọc danh sách series có `Modality == 'CT'` bằng Python.


In [ ]:
# 11. Filter CT series
ct_series = [
    s for s in all_series
    if str(s.get("Modality", "")).upper() == "CT"
]

print(f"Số lượng CT series tìm thấy: {len(ct_series)}")
for idx, s in enumerate(ct_series):
    size_mb = s.get("FileSize", 0) / (1024**2) if s.get("FileSize") else 0.0
    print(f"  [{idx}] Series UID: {s.get('SeriesInstanceUID')}")
    print(f"      Images: {s.get('ImageCount')} | Size: {size_mb:.2f} MB | Desc: {s.get('SeriesDescription')}")

if not ct_series:
    raise RuntimeError(f"Không tìm thấy CT series nào trong Study {study_uid}. Hãy kiểm tra lại SELECTED_STUDY_INDEX!")

print("\n[PASS] Xác định được CT series.")


---
## 12. Select series
Chọn một CT series để tải. Có kiểm tra và cảnh báo nếu `ImageCount` hoặc `FileSize` vượt ngưỡng giới hạn Colab.


In [ ]:
# 12. Select series
CT_SERIES_INDEX = 0
selected_ct = ct_series[CT_SERIES_INDEX]
selected_series_uid = selected_ct["SeriesInstanceUID"]
image_count = int(selected_ct.get("ImageCount", 0))
file_size_mb = (selected_ct.get("FileSize", 0) / (1024**2)) if selected_ct.get("FileSize") else 0.0

print(f"Series CT được chọn [index {CT_SERIES_INDEX}]:")
print("  SeriesInstanceUID:", selected_series_uid)
print(f"  Số lát cắt (images): {image_count}")
print(f"  Dung lượng dự kiến: {file_size_mb:.2f} MB")

# Kiểm tra ngưỡng an toàn cho Colab
if image_count > 600:
    print(f"⚠️ CẢNH BÁO: Series có {image_count} ảnh (> 600). Việc tải và xử lý có thể mất nhiều RAM/thời gian.")
if file_size_mb > 300:
    print(f"⚠️ CẢNH BÁO: Dung lượng {file_size_mb:.2f} MB tương đối lớn.")


---
## 13. Download series
Tải duy nhất 1 series ZIP từ endpoint `getImage` bằng `SeriesInstanceUID`. Lưu vào `/content/lidc_demo/raw/<series_uid>.zip`.


In [ ]:
# 13. Download series
zip_path = RAW_DIR / f"{selected_series_uid}.zip"

download_url = f"{BASE_URL}/getImage"
params = {"SeriesInstanceUID": selected_series_uid}

print(f"Bắt đầu tải series {selected_series_uid}...")
print("Endpoint:", download_url)

with requests.get(download_url, params=params, stream=True, timeout=600) as resp:
    resp.raise_for_status()
    total_size = int(resp.headers.get("content-length", 0))

    with open(zip_path, "wb") as f:
        with tqdm(
            total=total_size if total_size > 0 else None,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc=f"Download {patient_id} CT",
        ) as pbar:
            for chunk in resp.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

downloaded_size_mb = zip_path.stat().st_size / (1024**2)
print(f"\n✅ Đã tải file zip thành công: {zip_path} ({downloaded_size_mb:.2f} MB)")

assert zipfile.is_zipfile(zip_path), f"File {zip_path} không phải là file ZIP hợp lệ!"
print("[PASS] Download 1 series thành công và đã xác thực tính toàn vẹn của file ZIP.")


---
## 14. Extract DICOM
Giải nén file ZIP vào cấu trúc thư mục `/content/lidc_demo/dicom/<patient_id>/<series_uid>/`.


In [ ]:
# 14. Extract DICOM
series_extract_dir = EXTRACT_DIR / patient_id / selected_series_uid
series_extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(series_extract_dir)

all_extracted_files = list(series_extract_dir.rglob("*"))
print(f"Đã giải nén vào: {series_extract_dir}")
print(f"Tổng số file sau giải nén: {len(all_extracted_files)}")


---
## 15. Read DICOM
Quét đệ quy thư mục giải nén bằng `pydicom`. Không giả định định dạng tên file hay phần mở rộng; chỉ đọc những file có `hasattr(ds, 'PixelData')`.


In [ ]:
# 15. Read DICOM
dicom_objects = []

for p in series_extract_dir.rglob("*"):
    if p.is_file():
        try:
            ds = pydicom.dcmread(p, force=True)
            if hasattr(ds, "PixelData"):
                dicom_objects.append(ds)
        except Exception:
            pass

print(f"Số lượng file DICOM ảnh hợp lệ đọc được: {len(dicom_objects)}")
assert len(dicom_objects) > 0, "Không tìm thấy file DICOM nào có PixelData!"
print("[PASS] Đọc được DICOM thành công.")


---
## 16. Sort slices
Sắp xếp các lát cắt DICOM theo thứ tự không gian 3D dọc theo trục Z.
- Ưu tiên 1: `ImagePositionPatient[2]`
- Fallback 2: `SliceLocation`
- Fallback 3: `InstanceNumber`


In [ ]:
# 16. Sort slices
def get_slice_z(ds):
    ipp = getattr(ds, "ImagePositionPatient", None)
    if ipp is not None and len(ipp) >= 3:
        try:
            return float(ipp[2])
        except (ValueError, TypeError):
            pass

    loc = getattr(ds, "SliceLocation", None)
    if loc is not None:
        try:
            return float(loc)
        except (ValueError, TypeError):
            pass

    return float(getattr(ds, "InstanceNumber", 0))

# Sắp xếp các lát cắt theo toạ độ Z tăng dần
dicom_objects = sorted(dicom_objects, key=get_slice_z)

z_coords = [get_slice_z(ds) for ds in dicom_objects]
print(f"First Slice Z: {z_coords[0]:.2f} mm")
print(f"Last Slice Z : {z_coords[-1]:.2f} mm")
print(f"Z span       : {abs(z_coords[-1] - z_coords[0]):.2f} mm")
print(f"Slice step   : {abs(z_coords[1] - z_coords[0]):.2f} mm")

print("\n[PASS] Sort đúng slice theo trục Z.")


---
## 17. Build raw volume
Ghép các lát cắt 2D thành mảng thể tích 3D NumPy với quy ước chuẩn `(num_slices, height, width)` tức `volume[z, y, x]`. Không transpose tùy tiện.


In [ ]:
# 17. Build raw volume
volume_raw = np.stack([
    ds.pixel_array.astype(np.float32)
    for ds in dicom_objects
], axis=0)

print("Volume raw shape:", volume_raw.shape)
print("Volume raw dtype:", volume_raw.dtype)
print(f"Volume raw min  : {volume_raw.min():.1f}")
print(f"Volume raw max  : {volume_raw.max():.1f}")

assert volume_raw.ndim == 3, "Volume phải có đúng 3 chiều (Z, Y, X)!"
print("\n[PASS] Tạo được volume thô thành công.")


---
## 18. Convert to HU
Chuyển đổi giá trị pixel thô sang Hounsfield Units (HU) chuẩn:
$$\text{HU} = \text{pixel\_value} \times \text{RescaleSlope} + \text{RescaleIntercept}$$
Lấy giá trị slope và intercept từ metadata của từng lát cắt.


In [ ]:
# 18. Convert to HU
hu_slices = []

for idx, ds in enumerate(dicom_objects):
    pixel_data = volume_raw[idx].copy()
    slope = float(getattr(ds, "RescaleSlope", 1.0))
    intercept = float(getattr(ds, "RescaleIntercept", 0.0))

    slice_hu = pixel_data * slope + intercept
    hu_slices.append(slice_hu)

volume_hu = np.stack(hu_slices, axis=0).astype(np.float32)

print("--- Volume HU Statistics ---")
print("Shape:", volume_hu.shape, f"(Z={volume_hu.shape[0]}, Y={volume_hu.shape[1]}, X={volume_hu.shape[2]})")
print("Dtype:", volume_hu.dtype)
print(f"Min  : {volume_hu.min():.2f} HU")
print(f"Max  : {volume_hu.max():.2f} HU")
print(f"Mean : {volume_hu.mean():.2f} HU")
print(f"Std  : {volume_hu.std():.2f} HU")

print("\n[PASS] Convert HU thành công.")


---
## 19. Inspect metadata
Trích xuất và hiển thị đầy đủ các thông số siêu dữ liệu lâm sàng và hình học của ca chụp.


In [ ]:
# 19. Inspect metadata
ds0 = dicom_objects[0]

meta_fields = [
    ("PatientID", getattr(ds0, "PatientID", patient_id)),
    ("StudyInstanceUID", getattr(ds0, "StudyInstanceUID", study_uid)),
    ("SeriesInstanceUID", getattr(ds0, "SeriesInstanceUID", selected_series_uid)),
    ("Modality", getattr(ds0, "Modality", "CT")),
    ("SeriesDescription", getattr(ds0, "SeriesDescription", "None")),
    ("ImageCount", len(dicom_objects)),
    ("Rows", getattr(ds0, "Rows", None)),
    ("Columns", getattr(ds0, "Columns", None)),
    ("PixelSpacing", getattr(ds0, "PixelSpacing", None)),
    ("SliceThickness", getattr(ds0, "SliceThickness", None)),
    ("ImagePositionPatient", getattr(ds0, "ImagePositionPatient", None)),
    ("RescaleSlope", getattr(ds0, "RescaleSlope", None)),
    ("RescaleIntercept", getattr(ds0, "RescaleIntercept", None)),
    ("AnnotationsFlag", selected_ct.get("AnnotationsFlag", None)),
    ("BodyPartExamined", getattr(ds0, "BodyPartExamined", selected_ct.get("BodyPartExamined"))),
    ("StudyDate", getattr(ds0, "StudyDate", selected_study.get("StudyDate"))),
]

print("="*60)
print(f"{'DICOM METADATA SUMMARY':^60}")
print("="*60)
for key, val in meta_fields:
    print(f"  {key:25s}: {val}")
print("="*60)


---
## 20. Display single slice
Hiển thị lát cắt ở giữa thể tích (`mid_idx`) ở chế độ Raw HU.


In [ ]:
# 20. Display single slice (Raw HU)
mid_idx = volume_hu.shape[0] // 2

plt.figure(figsize=(8, 8))
plt.imshow(volume_hu[mid_idx], cmap="gray")
plt.title(f"{patient_id} — Slice {mid_idx} (Raw HU)\nRange: [{volume_hu[mid_idx].min():.1f}, {volume_hu[mid_idx].max():.1f} HU]", fontsize=12)
plt.axis("off")
plt.show()

print("[PASS] Hiển thị CT slice thành công.")


---
## 21. Display lung window
Hiển thị lát cắt phổi theo cửa sổ chuẩn mô phổi (**Lung Window**: `vmin = -1000 HU`, `vmax = +400 HU`). Đây là góc nhìn quan trọng nhất để bác sĩ và mô hình AI phát hiện tổn thương nhu mô và nốt phổi.


In [ ]:
# 21. Display lung window
plt.figure(figsize=(8, 8))
plt.imshow(
    volume_hu[mid_idx],
    cmap="gray",
    vmin=LUNG_WINDOW_VMIN,
    vmax=LUNG_WINDOW_VMAX
)
plt.title(f"{patient_id} — Slice {mid_idx} (Lung Window: -1000 to +400 HU)", fontsize=12)
plt.axis("off")
plt.show()

print("[PASS] Hiển thị lung window thành công.")


---
## 22. Display multiple slices
Hiển thị 12 lát cắt phân bố đều theo chiều dọc từ đỉnh đến đáy phổi (`slice 0, N/11, 2N/11, ..., N-1`) để khảo sát tổng thể cấu trúc 3D của lồng ngực.


In [ ]:
# 22. Display multiple slices
num_slices = volume_hu.shape[0]
indices = np.linspace(0, num_slices - 1, 12, dtype=int)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, idx in enumerate(indices):
    axes[i].imshow(
        volume_hu[idx],
        cmap="gray",
        vmin=LUNG_WINDOW_VMIN,
        vmax=LUNG_WINDOW_VMAX
    )
    axes[i].set_title(f"Slice {idx} (Z: {z_coords[idx]:.1f} mm)", fontsize=10)
    axes[i].axis("off")

fig.suptitle(f"{patient_id} — Volume Overview (12 slices, Lung Window)", fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

print("[PASS] Hiển thị 12 lát cắt phân bố đều thành công.")


---
## 23. Annotation discovery
Khảo sát và trích xuất Ground Truth Annotation nốt phổi:
1. **DICOM SEG & SR series**: Đã được phát hiện trong Study 1 ở Cell 10 từ các bác sĩ chẩn đoán hình ảnh.
2. **LIDC XML**: Tải kho dữ liệu XML từ Cancer Imaging Archive, trích xuất đặc trưng lâm sàng của nốt phổi (subtlety, calcification, malignancy,...), tọa độ đường viền đa giác contour, và tạo mặt nạ nhị phân (Ground Truth Mask) đè lên lát cắt CT.


In [ ]:
# 23. Annotation discovery — Tải và phân tích LIDC XML
import xml.etree.ElementTree as ET
from matplotlib.path import Path as MplPath

xml_zip_url = "https://www.cancerimagingarchive.net/wp-content/uploads/LIDC-XML-only.zip"
xml_zip_path = Path("/content/LIDC-XML-only.zip")

if not xml_zip_path.exists():
    print(f"Đang tải LIDC XML annotations từ {xml_zip_url}...")
    xml_resp = requests.get(xml_zip_url, timeout=600)
    xml_resp.raise_for_status()
    xml_zip_path.write_bytes(xml_resp.content)
    print(f"Đã lưu {xml_zip_path} ({len(xml_resp.content)/(1024**2):.2f} MB)")

with zipfile.ZipFile(xml_zip_path, "r") as z:
    z.extractall(XML_DIR)

all_xml_files = list(XML_DIR.rglob("*.xml"))
print(f"Tổng số file XML giải nén: {len(all_xml_files)}")

# Tìm file XML chính xác qua SeriesInstanceUID (hoặc StudyInstanceUID / patient_id)
# Trong kho LIDC-XML-only, file XML được đặt tên theo lô (vd: tcia-lidc-xml/185/069.xml)
# và chứa SeriesInstanceUid trong thẻ <ResponseHeader>
print(f"Đang tìm XML annotation khớp với SeriesInstanceUID: {selected_series_uid}...")
matched_xml = []
for f in all_xml_files:
    try:
        content = f.read_text(encoding="utf-8", errors="ignore")
        if selected_series_uid in content and "unblindedReadNodule" in content:
            matched_xml.append(f)
    except Exception:
        pass

if not matched_xml:
    for f in all_xml_files:
        try:
            content = f.read_text(encoding="utf-8", errors="ignore")
            if selected_series_uid in content:
                matched_xml.append(f)
        except Exception:
            pass

if not matched_xml:
    # Fallback tìm theo study_uid nếu có
    if "study_uid" in globals():
        for f in all_xml_files:
            try:
                content = f.read_text(encoding="utf-8", errors="ignore")
                if study_uid in content:
                    matched_xml.append(f)
            except Exception:
                pass

if not matched_xml:
    # Fallback cuối theo tên file
    match = re.search(r'(\d+)$', patient_id)
    numeric_id = match.group(1) if match else "0001"
    matched_xml = [f for f in all_xml_files if numeric_id in f.name]

assert len(matched_xml) > 0, f"Không tìm thấy file XML nào cho series {selected_series_uid}!"
xml_path = matched_xml[0]
print(f"✅ Đã tìm thấy XML annotation: {xml_path.relative_to(XML_DIR)}")

# Hàm hỗ trợ đọc thẻ XML độc lập với namespace (vd: xmlns='http://www.nih.gov')
def get_tag(el):
    return el.tag.split("}")[-1] if "}" in el.tag else el.tag

def find_children(parent, tag):
    return [c for c in parent if get_tag(c).lower() == tag.lower()]

def find_child(parent, tag):
    children = find_children(parent, tag)
    return children[0] if children else None

def get_text(parent, tag):
    c = find_child(parent, tag)
    return c.text.strip() if (c is not None and c.text) else None

tree = ET.parse(xml_path)
root = tree.getroot()

# Tìm tất cả unblindedReadNodule (nốt phổi >= 3mm)
nodules = [el for el in root.iter() if get_tag(el).lower() == "unblindedreadnodule"]
print(f"Số lượng nốt phổi (>= 3mm) được đánh dấu bởi các bác sĩ: {len(nodules)}")

if nodules:
    # Chọn nốt có nhiều ROI nhất để hiển thị trực quan rõ nét nhất
    best_nodule = max(nodules, key=lambda n: len([c for c in n.iter() if get_tag(c).lower() == "roi"]))
    nodule_id = get_text(best_nodule, "noduleID") or "Nodule"

    print(f"\nĐặc trưng lâm sàng của nốt phổi ({nodule_id}):")
    chars = find_child(best_nodule, "characteristics")
    if chars is not None:
        for feat in ["subtlety", "internalStructure", "calcification", "sphericity", "margin", "lobulation", "spiculation", "texture", "malignancy"]:
            val = get_text(chars, feat)
            print(f"  {feat:20s}: {val}")

    # Lấy danh sách ROI
    roi_list = [c for c in best_nodule.iter() if get_tag(c).lower() == "roi"]
    print(f"  Số ROI / lát cắt có nốt: {len(roi_list)}")

    if roi_list:
        # Chọn lát cắt ở giữa nốt phổi
        roi = roi_list[len(roi_list) // 2]
        z_txt = get_text(roi, "imageZposition")
        z_nodule = float(z_txt) if z_txt else 0.0

        # Khớp với lát cắt CT có tọa độ Z gần nhất
        z_arr = np.array(z_coords)
        matched_slice_idx = int(np.argmin(np.abs(z_arr - z_nodule)))
        print(f"  Z nốt phổi: {z_nodule:.2f} mm -> Khớp với CT Slice Index: {matched_slice_idx} (Slice Z: {z_coords[matched_slice_idx]:.2f} mm)")

        # Trích xuất contour points
        edge_maps = [c for c in roi.iter() if get_tag(c).lower() == "edgemap"]
        points = []
        for edge in edge_maps:
            x_txt = get_text(edge, "xCoord")
            y_txt = get_text(edge, "yCoord")
            if x_txt and y_txt:
                points.append((float(x_txt), float(y_txt)))
        points = np.array(points)
        print(f"  Số điểm đỉnh đường bao (contour vertices): {len(points)}")

        # Tạo boolean mask ground truth
        h, w = volume_hu[matched_slice_idx].shape
        yy, xx = np.mgrid[0:h, 0:w]
        coords = np.column_stack([xx.ravel(), yy.ravel()])
        poly_path = MplPath(points)
        mask = poly_path.contains_points(coords).reshape(h, w)
        print(f"  Số pixel nốt phổi trong mask: {mask.sum()}")

        # Hiển thị Contour & Ground Truth Mask
        plt.figure(figsize=(10, 10))
        plt.imshow(volume_hu[matched_slice_idx], cmap="gray", vmin=LUNG_WINDOW_VMIN, vmax=LUNG_WINDOW_VMAX)
        plt.plot(points[:, 0], points[:, 1], color="cyan", linewidth=2.5, label="Nodule Contour")
        plt.scatter(points[:, 0], points[:, 1], color="yellow", s=15, zorder=3, label="Contour Points")
        plt.imshow(np.ma.masked_where(~mask, mask), cmap="autumn", alpha=0.45, label="Nodule Mask")
        plt.title(f"{patient_id} — Ground Truth Nodule Annotation (Slice {matched_slice_idx})", fontsize=12)
        plt.legend(loc="upper right")
        plt.axis("off")
        plt.show()

print("\n[PASS] Hoàn thành toàn bộ pipeline đọc dữ liệu LIDC-IDRI trên Google Colab!")


---
## 24. Multi-Slice Nodule 3D Progression (Khảo sát chuỗi 30 lát cắt liên tiếp)
Khảo sát sự hình thành, phát triển và kết thúc của nốt phổi và mặt nạ phân đoạn (Ground Truth Mask & Contour) qua **30 lát cắt CT liên tiếp dọc theo trục Z**:
1. **Xây dựng mặt nạ 3D hoàn chỉnh**: Trích xuất toàn bộ các ROI và đường bao do bác sĩ vẽ trên mọi lát cắt có nốt phổi.
2. **Khung nhìn toàn cảnh lồng ngực (Whole-Lung 30-Slice Overview - 5x6 grid)**: Quan sát vị trí nốt trong giải phẫu lồng ngực, kèm bounding box định vị màu đỏ và contour/mask.
3. **Khung nhìn cận cảnh phóng to (Zoomed-in Nodule ROI Progression - 5x6 grid)**: Cắt crop cố định 80x80 pixel quanh tâm nốt qua 30 lát cắt liên tiếp, giúp thấy rõ từng chi tiết vi mô (ranh giới nốt, bờ nham nhở, tua gai) và quá trình nốt xuất hiện, phình to rồi biến mất.
4. **Mặt cắt trực giao 3 chiều (Multi-Planar Reconstruction: Axial, Coronal, Sagittal)**: Tái tạo nốt phổi trong không gian 3D qua 3 mặt phẳng trực giao kèm mặt nạ Ground Truth 3D.
5. **Bảng thống kê định lượng theo từng lát cắt**: Thống kê Slice Index, độ sâu Z (mm), số pixel diện tích nốt và trạng thái nốt.


In [ ]:
# 24.1 Xây dựng Ground Truth 3D Mask & Contours từ toàn bộ ROI của nốt phổi
from collections import defaultdict
import matplotlib.patches as patches
import pandas as pd

if "nodules" not in globals() or not nodules:
    print("⚠️ Không tìm thấy nốt phổi nào trong XML để trích xuất đa lát cắt!")
else:
    # Thu thập toàn bộ ROI của nốt phổi (best_nodule) trên từng lát cắt
    z_arr = np.array(z_coords)
    sop_to_idx = {getattr(ds, "SOPInstanceUID", ""): idx for idx, ds in enumerate(dicom_objects)}

    nodule_rois_by_slice = defaultdict(list)
    nodule_3d_mask = np.zeros_like(volume_hu, dtype=bool)

    h, w = volume_hu.shape[1], volume_hu.shape[2]
    yy, xx = np.mgrid[0:h, 0:w]
    all_pixel_coords = np.column_stack([xx.ravel(), yy.ravel()])

    roi_elements = [c for c in best_nodule.iter() if get_tag(c).lower() == "roi"]
    print(f"📌 Tổng số ROI được bác sĩ đánh dấu trên nốt {nodule_id}: {len(roi_elements)}")

    all_nodule_points = []

    for r in roi_elements:
        # 1. Tìm lát cắt khớp qua SOPInstanceUID hoặc Z position
        matched_idx = None
        sop_uid = get_text(r, "imageSOP_UID")
        if sop_uid and sop_uid in sop_to_idx:
            matched_idx = sop_to_idx[sop_uid]
        else:
            z_txt = get_text(r, "imageZposition")
            if z_txt:
                z_val = float(z_txt)
                matched_idx = int(np.argmin(np.abs(z_arr - z_val)))
                
        if matched_idx is None:
            continue
            
        # 2. Lấy danh sách điểm biên (edgeMap)
        edges = [c for c in r.iter() if get_tag(c).lower() == "edgemap"]
        pts = []
        for edge in edges:
            x_txt = get_text(edge, "xCoord")
            y_txt = get_text(edge, "yCoord")
            if x_txt and y_txt:
                pts.append((float(x_txt), float(y_txt)))
                
        if len(pts) >= 3:
            pts = np.array(pts)
            all_nodule_points.append(pts)
            poly = MplPath(pts)
            mask_2d = poly.contains_points(all_pixel_coords).reshape(h, w)
            nodule_3d_mask[matched_idx] |= mask_2d
            nodule_rois_by_slice[matched_idx].append({
                "points": pts,
                "mask": mask_2d,
                "z": z_coords[matched_idx],
                "area_pixels": int(mask_2d.sum())
            })

    slices_with_nodule = sorted(list(nodule_rois_by_slice.keys()))
    print(f"✅ Nốt phổi xuất hiện trên {len(slices_with_nodule)} lát cắt: {slices_with_nodule}")
    print(f"✅ Tổng thể tích nốt (voxel count): {nodule_3d_mask.sum():,} voxels")

    # Xác định tâm và hộp bao không gian 3D của nốt
    if all_nodule_points:
        all_pts_concat = np.concatenate(all_nodule_points, axis=0)
        x_min, y_min = np.min(all_pts_concat, axis=0)
        x_max, y_max = np.max(all_pts_concat, axis=0)
        center_x = int(round((x_min + x_max) / 2))
        center_y = int(round((y_min + y_max) / 2))
    else:
        x_min, x_max, y_min, y_max = 200, 300, 200, 300
        center_x, center_y = 256, 256

    min_slice = min(slices_with_nodule) if slices_with_nodule else volume_hu.shape[0] // 2
    max_slice = max(slices_with_nodule) if slices_with_nodule else volume_hu.shape[0] // 2
    center_slice = (min_slice + max_slice) // 2
    print(f"🎯 Tâm nốt phổi: Voxel (Z={center_slice}, Y={center_y}, X={center_x}) | Tọa độ Z = {z_coords[center_slice]:.2f} mm")

    # Chọn 30 lát cắt liên tiếp sao cho nốt phổi nằm ở trung tâm chuỗi
    num_total_slices = volume_hu.shape[0]
    target_num_slices = 30
    half_window = target_num_slices // 2

    start_slice = max(0, center_slice - half_window)
    end_slice = min(num_total_slices, start_slice + target_num_slices)
    if end_slice - start_slice < target_num_slices:
        start_slice = max(0, end_slice - target_num_slices)

    selected_30_indices = list(range(start_slice, end_slice))
    print(f"📊 Dải 30 lát cắt liên tiếp: Từ Slice {start_slice} đến Slice {end_slice - 1} (Tổng cộng: {len(selected_30_indices)} lát cắt)")

    # Bảng thống kê chi tiết lát cắt
    slice_stats = []
    ds0 = dicom_objects[0]
    pixel_spacing = getattr(ds0, "PixelSpacing", [0.7, 0.7])
    pixel_area_mm2 = float(pixel_spacing[0]) * float(pixel_spacing[1])

    for s_idx in selected_30_indices:
        has_nod = s_idx in nodule_rois_by_slice
        area_px = sum(r["area_pixels"] for r in nodule_rois_by_slice[s_idx]) if has_nod else 0
        area_mm2 = area_px * pixel_area_mm2
        slice_stats.append({
            "Slice_Index": s_idx,
            "Z_Coord_mm": round(z_coords[s_idx], 2),
            "Status": "🔴 Nodule Present" if has_nod else "⚪ Normal Lung",
            "Nodule_Pixels": area_px,
            "Area_mm2": round(area_mm2, 2)
        })

    df_30 = pd.DataFrame(slice_stats)
    print("\n--- Thống kê 30 lát cắt liên tiếp quanh nốt phổi ---")
    print(df_30.to_string(index=False))


### 24.2. Chuỗi 30 lát cắt toàn cảnh lồng ngực (Whole-Lung 5x6 Grid)
Hiển thị lưới 5x6 gồm 30 lát cắt liên tiếp trong cửa sổ phổi (**Lung Window**: `vmin = -1000 HU`, `vmax = +400 HU`).
- **Contour**: Đường biên đa giác màu xanh cyan và đỉnh vàng do các bác sĩ vẽ.
- **Mask**: Vùng phủ bán trong suốt màu đỏ cam thể hiện nốt ung thư.
- **Bounding Box**: Khung chữ nhật nét đứt màu đỏ giúp xác định ngay lập tức vị trí tổn thương trên toàn bộ trường phổi.


In [ ]:
# 24.2. Hiển thị 30 lát cắt toàn cảnh lồng ngực (Whole-Lung 5x6 Grid)
fig, axes = plt.subplots(5, 6, figsize=(22, 18))
axes = axes.flatten()

pad = 25  # Bán kính đệm cho hộp định vị quanh nốt

for i, s_idx in enumerate(selected_30_indices):
    ax = axes[i]
    ax.imshow(volume_hu[s_idx], cmap="gray", vmin=LUNG_WINDOW_VMIN, vmax=LUNG_WINDOW_VMAX)
    
    if s_idx in nodule_rois_by_slice:
        # 1. Vẽ đường viền và đỉnh contour
        for roi_data in nodule_rois_by_slice[s_idx]:
            pts = roi_data["points"]
            ax.plot(pts[:, 0], pts[:, 1], color="cyan", linewidth=1.8)
            ax.scatter(pts[:, 0], pts[:, 1], color="yellow", s=8, zorder=3)
            
        # 2. Phủ mặt nạ segmentation bán trong suốt
        m = nodule_3d_mask[s_idx]
        ax.imshow(np.ma.masked_where(~m, m), cmap="autumn", alpha=0.55)
        
        # 3. Vẽ hộp định vị màu đỏ giúp nhận biết vị trí nốt ngay lập tức
        rect = patches.Rectangle(
            (max(0, x_min - pad), max(0, y_min - pad)),
            (x_max - x_min + 2 * pad), (y_max - y_min + 2 * pad),
            linewidth=1.4, edgecolor="red", facecolor="none", linestyle="--"
        )
        ax.add_patch(rect)
        
        area = sum(r["area_pixels"] for r in nodule_rois_by_slice[s_idx])
        ax.set_title(f"Sl {s_idx} ({z_coords[s_idx]:.1f}mm)\n🔴 NODULE: {area} px", color="red", fontsize=9, fontweight="bold")
    else:
        ax.set_title(f"Sl {s_idx} ({z_coords[s_idx]:.1f}mm)\n⚪ Normal Lung", color="black", fontsize=8)
        
    ax.axis("off")

fig.suptitle(f"{patient_id} — Chuỗi 30 lát cắt CT liên tiếp dọc trục Z (Whole-Lung 5x6 Grid)", fontsize=16, y=0.99, fontweight="bold")
plt.tight_layout()
plt.show()


### 24.3. Khung nhìn cận cảnh phóng to (Zoomed-in Nodule ROI Progression - 5x6 Grid)
Phóng to cửa sổ 80x80 pixel quanh nốt phổi qua 30 lát cắt để quan sát chi tiết:
- **Giai đoạn trước nốt**: Nhu mô và mạch máu phổi hoàn toàn bình thường.
- **Giai đoạn khởi phát**: Nốt bắt đầu xuất hiện dưới dạng đám mờ nhỏ ở các lát cắt đầu tiên.
- **Giai đoạn cực đại**: Nốt phình to nhất, thể hiện rõ đường biên đa giác (cyan), các gai tua (*spiculation*), độ cản quang HU và cấu trúc bên trong.
- **Giai đoạn thu nhỏ**: Nốt co nhỏ dần và biến mất, trả lại nhu mô phổi bình thường.


In [ ]:
# 24.3. Khung nhìn cận cảnh phóng to (Zoomed-in Nodule ROI Progression)
crop_size = 80
half_crop = crop_size // 2

crop_x1 = max(0, center_x - half_crop)
crop_x2 = min(w, crop_x1 + crop_size)
crop_y1 = max(0, center_y - half_crop)
crop_y2 = min(h, crop_y1 + crop_size)

fig, axes = plt.subplots(5, 6, figsize=(20, 17))
axes = axes.flatten()

for i, s_idx in enumerate(selected_30_indices):
    ax = axes[i]
    crop_img = volume_hu[s_idx, crop_y1:crop_y2, crop_x1:crop_x2]
    ax.imshow(crop_img, cmap="gray", vmin=LUNG_WINDOW_VMIN, vmax=LUNG_WINDOW_VMAX)
    
    if s_idx in nodule_rois_by_slice:
        for roi_data in nodule_rois_by_slice[s_idx]:
            pts = roi_data["points"].copy()
            # Dịch chuyển tọa độ vào hệ quy chiếu của ảnh crop
            pts[:, 0] -= crop_x1
            pts[:, 1] -= crop_y1
            ax.plot(pts[:, 0], pts[:, 1], color="cyan", linewidth=2.0)
            ax.scatter(pts[:, 0], pts[:, 1], color="yellow", s=14, zorder=3)
            
        crop_m = nodule_3d_mask[s_idx, crop_y1:crop_y2, crop_x1:crop_x2]
        ax.imshow(np.ma.masked_where(~crop_m, crop_m), cmap="autumn", alpha=0.55)
        
        area = sum(r["area_pixels"] for r in nodule_rois_by_slice[s_idx])
        ax.set_title(f"Sl {s_idx} ({z_coords[s_idx]:.1f}mm)\n🔴 Nodule: {area} px", color="red", fontsize=9, fontweight="bold")
    else:
        ax.set_title(f"Sl {s_idx} ({z_coords[s_idx]:.1f}mm)\n⚪ Clean Parenchyma", color="black", fontsize=8)
        
    ax.axis("off")

fig.suptitle(f"{patient_id} — Cận cảnh phóng to (80x80) nốt phổi qua 30 lát cắt liên tiếp", fontsize=16, y=0.99, fontweight="bold")
plt.tight_layout()
plt.show()


### 24.4. Mặt cắt trực giao 3 chiều (Multi-Planar Reconstruction: Axial, Coronal, Sagittal)
Tái tạo nốt phổi trong không gian 3D qua 3 mặt phẳng trực giao kèm mặt nạ Ground Truth 3D để hình dung trọn vẹn thể tích hình học của nốt trong lồng ngực.


In [ ]:
# 24.4. Mặt cắt trực giao 3 chiều (Multi-Planar Reconstruction - MPR)
ds0 = dicom_objects[0]
pixel_spacing = getattr(ds0, "PixelSpacing", [0.7, 0.7])
slice_thickness = getattr(ds0, "SliceThickness", 2.5)
if abs(z_coords[1] - z_coords[0]) > 0:
    slice_thickness = abs(z_coords[1] - z_coords[0])

spacing_z = float(slice_thickness)
spacing_y = float(pixel_spacing[0])
spacing_x = float(pixel_spacing[1])

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# 1. Axial View (Mặt phẳng ngang Z: Nhìn từ dưới lên)
ax = axes[0]
ax.imshow(volume_hu[center_slice], cmap="gray", vmin=LUNG_WINDOW_VMIN, vmax=LUNG_WINDOW_VMAX)
m_ax = nodule_3d_mask[center_slice]
if m_ax.any():
    ax.imshow(np.ma.masked_where(~m_ax, m_ax), cmap="autumn", alpha=0.5)
ax.axhline(center_y, color='yellow', linestyle='--', linewidth=0.9, alpha=0.7)
ax.axvline(center_x, color='yellow', linestyle='--', linewidth=0.9, alpha=0.7)
ax.set_title(f"Axial View (Z = {center_slice}, Z={z_coords[center_slice]:.1f}mm)", fontsize=12, fontweight="bold")
ax.axis("off")

# 2. Coronal View (Mặt phẳng đứng ngang Y: Nhìn từ trước ra sau)
ax = axes[1]
cor_img = volume_hu[:, center_y, :]
m_cor = nodule_3d_mask[:, center_y, :]
aspect_cor = spacing_z / spacing_y
ax.imshow(cor_img, cmap="gray", vmin=LUNG_WINDOW_VMIN, vmax=LUNG_WINDOW_VMAX, aspect=aspect_cor, origin="lower")
if m_cor.any():
    ax.imshow(np.ma.masked_where(~m_cor, m_cor), cmap="autumn", alpha=0.5, aspect=aspect_cor, origin="lower")
ax.axhline(center_slice, color='cyan', linestyle='--', linewidth=0.9, alpha=0.7)
ax.axvline(center_x, color='cyan', linestyle='--', linewidth=0.9, alpha=0.7)
ax.set_title(f"Coronal View (Y = {center_y})", fontsize=12, fontweight="bold")
ax.axis("off")

# 3. Sagittal View (Mặt phẳng đứng dọc X: Nhìn từ bên hông)
ax = axes[2]
sag_img = volume_hu[:, :, center_x]
m_sag = nodule_3d_mask[:, :, center_x]
aspect_sag = spacing_z / spacing_x
ax.imshow(sag_img, cmap="gray", vmin=LUNG_WINDOW_VMIN, vmax=LUNG_WINDOW_VMAX, aspect=aspect_sag, origin="lower")
if m_sag.any():
    ax.imshow(np.ma.masked_where(~m_sag, m_sag), cmap="autumn", alpha=0.5, aspect=aspect_sag, origin="lower")
ax.axhline(center_slice, color='magenta', linestyle='--', linewidth=0.9, alpha=0.7)
ax.axvline(center_y, color='magenta', linestyle='--', linewidth=0.9, alpha=0.7)
ax.set_title(f"Sagittal View (X = {center_x})", fontsize=12, fontweight="bold")
ax.axis("off")

plt.suptitle(f"{patient_id} — Multi-Planar Reconstruction (MPR) với 3D Ground Truth Mask", fontsize=15, y=0.98, fontweight="bold")
plt.tight_layout()
plt.show()

print("\n[PASS] Hoàn thành hiển thị 30 lát cắt liên tiếp và tái tạo không gian 3D của nốt phổi!")


### 24.5. Thanh trượt tương tác thời gian thực (Interactive Slice Scrubbing)
Kéo thanh trượt để duyệt qua từng lát cắt bất kỳ trong toàn bộ ca chụp CT. Lát cắt nào có nốt phổi sẽ tự động hiển thị tâm nốt, contour và mask.


In [ ]:
# 24.5. Thanh trượt tương tác thời gian thực (Interactive Slider)
from ipywidgets import interact, IntSlider

def view_interactive_slice(slice_num):
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(volume_hu[slice_num], cmap="gray", vmin=LUNG_WINDOW_VMIN, vmax=LUNG_WINDOW_VMAX)
    
    if slice_num in nodule_rois_by_slice:
        for roi_data in nodule_rois_by_slice[slice_num]:
            pts = roi_data["points"]
            ax.plot(pts[:, 0], pts[:, 1], color="cyan", linewidth=2.0)
            ax.scatter(pts[:, 0], pts[:, 1], color="yellow", s=15, zorder=3)
        m = nodule_3d_mask[slice_num]
        ax.imshow(np.ma.masked_where(~m, m), cmap="autumn", alpha=0.5)
        area = sum(r["area_pixels"] for r in nodule_rois_by_slice[slice_num])
        ax.set_title(f"Slice {slice_num}/{volume_hu.shape[0]-1} (Z={z_coords[slice_num]:.1f}mm) — 🔴 NODULE ({area} px)", color="red", fontsize=12, fontweight="bold")
    else:
        ax.set_title(f"Slice {slice_num}/{volume_hu.shape[0]-1} (Z={z_coords[slice_num]:.1f}mm) — ⚪ Normal Lung", fontsize=12)
        
    ax.axis("off")
    plt.show()

interact(
    view_interactive_slice,
    slice_num=IntSlider(min=0, max=volume_hu.shape[0]-1, step=1, value=center_slice, description="CT Slice:")
);
